# 00.03 · Jupyter fundamentals

**Module 00 · Foundations, Tooling & Research Skills** · Part 1 (Core)

| | |
|---|---|
| ⏱️ Estimated time | 60–90 min |
| 📋 Prerequisites | Lessons 00.01–00.02 and the [create-course-env guide](00-02-create-course-env.md). |
| 🔑 Keys | None |
| 🧠 Kernel | **Python (lc-mastery)**. Check the top-right corner of the notebook. |
| ✅ Docs verified | 2026-10-11 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**.
> Run cells **from top to bottom**. If anything looks strange, use **Kernel → Restart Kernel and Run All Cells…**

## Learning objectives

1. **Navigate** Jupyter Notebook 7 (and JupyterLab 4) with the mouse and the most useful keyboard shortcuts.
2. **Explain** what the kernel is, why its memory ("state") survives between cells, and how that causes out-of-order bugs.
3. **Use** *Restart Kernel and Run All Cells* as the "does it really work?" test before every commit.
4. **Use** core IPython magics (`%time`, `%%time`, `%timeit`, `%who`, `%env`, `?`) and know which ones can leak secrets.
5. **Describe** the `.ipynb` file format and why outputs must be cleared before committing.

## 📚 Official reading (verified 2026-10-11)

| Page | Why |
|---|---|
| [Running the Notebook](https://docs.jupyter.org/en/latest/running.html) | Starting Jupyter, the dashboard, stopping the server |
| [The Jupyter Notebook](https://jupyter-notebook.readthedocs.io/en/stable/notebook.html) | Cells, modes, kernels, keyboard shortcuts, trusting notebooks |
| [JupyterLab: notebooks](https://jupyterlab.readthedocs.io/en/stable/user/notebook.html) · [JupyterLab commands and shortcuts](https://jupyterlab.readthedocs.io/en/stable/user/commands.html) | Notebook 7 is built on JupyterLab, so the same commands apply |
| [IPython tutorial](https://ipython.readthedocs.io/en/stable/interactive/tutorial.html) · [Built-in magics](https://ipython.readthedocs.io/en/stable/interactive/magics.html) | Magics, `?` help, history |
| [nbformat: the notebook file format](https://nbformat.readthedocs.io/en/latest/format_description.html) | What is inside an `.ipynb` file |
| [nbstripout](https://github.com/kynan/nbstripout) | Strip outputs automatically on `git add` (bonus) |

## Setup: confirm the kernel

Every lesson from now on runs on the course kernel. The cell below fails loudly if you are on the wrong one, because a wrong kernel causes confusing errors later.

In [ ]:
import sys
from pathlib import Path

prefix = Path(sys.prefix)
print("Kernel Python:", sys.executable)

# WHY a check instead of an error: the lesson still works in any Python 3 kernel,
# but the next lessons install packages, and they must go into lc-mastery.
if prefix.name == "lc-mastery" or (prefix / "pyvenv.cfg").is_file():
    print("✅ Course environment detected.")
else:
    print("⚠️ This is not the lc-mastery environment. Use Kernel → Change Kernel… → Python (lc-mastery).")
    print("   No such kernel? Follow 00-02-create-course-env.md first.")

---
## 1. A quick tour of the interface

| Area | What it does |
|---|---|
| **File browser** (the "dashboard", first browser tab) | Opens notebooks, creates folders and files, shows which notebooks are *running* (green icon) |
| **Menu bar** | *File, Edit, View, Run, Kernel, Settings, Help*. Every action lives here, so you never *need* a shortcut |
| **Toolbar** | Save, insert cell, cut/copy/paste, run ▶, interrupt ■, restart ⟳, cell type dropdown |
| **Kernel indicator** (top right) | Kernel name, plus a circle: hollow = idle, filled = busy |
| **Command palette** | **Ctrl + Shift + C**: search any command by name (great when you forget a shortcut) |

### Two modes

Jupyter has **command mode** (you act on whole cells: blue bar on the left, no cursor) and **edit mode** (you type inside a cell).

- **Esc** → command mode. **Enter** → edit mode.

### Shortcuts worth learning today

Verified against the [JupyterLab command list](https://jupyterlab.readthedocs.io/en/stable/user/commands.html) (Notebook 7 uses the same ones). Command-mode shortcuts are single keys, so press **Esc** first.

| Shortcut | Mode | Action |
|---|---|---|
| **Shift + Enter** | any | Run cell, move to the next one |
| **Ctrl + Enter** | any | Run cell, stay on it |
| **Alt + Enter** | any | Run cell, insert a new cell below |
| **A** / **B** | command | Insert cell **a**bove / **b**elow |
| **D, D** (press twice) | command | Delete cell |
| **Z** | command | Undo the last cell operation (e.g., an accidental delete) |
| **M** / **Y** / **R** | command | Make cell Markdown / code / raw |
| **C**, **X**, **V** | command | Copy, cut, paste cells |
| **Shift + M** | command | Merge selected cells |
| **Ctrl + Shift + -** | edit | Split cell at the cursor |
| **I, I** | command | Interrupt the kernel (stop a running cell) |
| **0, 0** | command | Restart the kernel |
| **Ctrl + S** | any | Save |

The full list is in **Help → Keyboard Shortcuts** (as the [Notebook docs](https://jupyter-notebook.readthedocs.io/en/stable/notebook.html) describe it) and in *Settings → Settings Editor → Keyboard Shortcuts*, where you can also add your own.

### Practice (2 minutes)

1. Select this cell (click the left margin), press **B** to add a cell below.
2. Press **M** to make it Markdown, **Enter** to edit, type `## My first note`, press **Shift + Enter**.
3. Select your new cell, press **D, D** to delete it, then **Z** to bring it back.

---
## 2. The kernel remembers everything (state)

When you run a cell, the kernel executes it and **keeps every variable in memory** until the kernel stops. The notebook is just a view: the *order you ran cells in* matters, not the order they appear on the page.

Run the next two cells. Then run the **second** one three more times.

In [ ]:
counter = 0
print("counter was reset to", counter)

In [ ]:
counter += 1
print("counter is now", counter)

### What just happened?

- `counter` lives in the kernel's memory. Each run of the second cell added 1 to the *current* value, so you saw 1, 2, 3, 4.
- The number in `[ ]` to the left of each cell is its **execution count**. It goes up every time *any* cell runs. If the counts on the page are not in increasing order from top to bottom, the notebook was run out of order.

### The out-of-order bug

Here is the classic trap. Run the three cells below **in order**.

In [ ]:
discount = 0.10   # 10 % off

In [ ]:
price = 100 * (1 - discount)
print(f"Price after discount: {price:.2f}")

In [ ]:
discount = 0.50   # Someone changed the discount further down the notebook...

Now go back and run the **`price = ...` cell again** (click it, **Ctrl + Enter**). It prints `50.00`, not `90.00`, although the cell *directly above it* still says `0.10`.

That is **hidden state**: the result depends on the history of what you ran, which nobody can see by reading the page. Your teammate (or you next week) will get different results.

**Inspect the kernel's memory** with the `%who` and `%whos` magics:

In [ ]:
%whos

`%whos` lists every variable you created, with its type and value. Deleting a cell does **not** delete its variables: they stay in memory until you `del` them or restart the kernel.

---
## 3. Restart & Run All: the "does it really work?" test

The only reliable way to prove a notebook works is to **start from an empty kernel and run every cell from top to bottom**:

**Kernel → Restart Kernel and Run All Cells…** (toolbar: ⏩ in Notebook 7), then confirm **Restart**.

**You should now see** execution counts `[1]`, `[2]`, `[3]`… increasing from top to bottom with no gaps, and the `price` cell printing `90.00` again, because the cells now ran in page order.

The course rule ([CONVENTIONS.md §3](../../CONVENTIONS.md#3-lesson-notebook-structure-template)): **Restart & Run All must pass before every commit.**

You can also check the execution count from code:

In [ ]:
from IPython import get_ipython

ip = get_ipython()  # The running IPython shell (None outside Jupyter/IPython).
print("This cell's execution count:", ip.execution_count if ip else "not running in IPython")
print("Cells run so far in this kernel session:", len(ip.user_ns["In"]) - 1 if ip else "n/a")

---
## 4. Stopping a cell that runs too long

The next cell simulates slow work for about 5 seconds and prints progress. Run it and, while it runs, press **Esc** then **I, I** (or click ■ in the toolbar) to **interrupt** it. Python raises `KeyboardInterrupt`, and we catch it to stop cleanly.

(If you don't interrupt it, it simply finishes. Both are fine.)

In [ ]:
import time

STEPS = 10
try:
    for step in range(1, STEPS + 1):
        time.sleep(0.5)                       # Pretend this is a slow API call.
        print(f"step {step}/{STEPS}", end="\r")  # \r rewrites the same line: tidy progress.
    print("\nDone: all steps finished.")
except KeyboardInterrupt:
    # WHY catch it: an interrupted cell should leave a clear message, not a scary traceback.
    print(f"\nInterrupted at step {step}. The kernel is still alive; your variables are kept.")

**Interrupt vs restart:** *interrupt* stops the current cell and **keeps** all variables. *Restart* (**0, 0**) kills the kernel and **forgets** everything. Use interrupt first; restart when the kernel is stuck or memory is full.

---
## 5. Magics: IPython's built-in superpowers

Lines starting with `%` (line magics) or cells starting with `%%` (cell magics) are commands for IPython, not Python. `%lsmagic` lists them all.

### Timing code

In [ ]:
# %time runs a statement once and reports how long it took.
%time total = sum(range(10_000_000))

In [ ]:
%%time
# %%time must be the FIRST line of the cell. It times the whole cell.
squares = [n * n for n in range(1_000_000)]
print(len(squares), "squares computed")

In [ ]:
# %timeit runs a statement many times and reports the average: better for small, fast snippets.
%timeit -n 1000 -r 3 sorted([5, 3, 1, 4, 2])

You will use `%%time` constantly to measure model calls (Lesson 00.07 and Project 00).

### Getting help with `?`

Put `?` after any object to see its documentation; `??` also shows the source code when available. In Notebook 7 the help opens in a panel at the bottom. **Shift + Tab** inside a function call's parentheses shows a quick tooltip.

In [ ]:
sorted?

### Environment variables with `%env` (careful!)

`%env NAME=value` sets a variable for this kernel; `%env NAME` reads one.

In [ ]:
%env MASTERY_DEMO_FLAG=on
%env MASTERY_DEMO_FLAG

> 🚨 **Never run a bare `%env` (without a name) in a course notebook.** It prints **every** environment variable, including API keys loaded from `.env`, and that output is saved inside the `.ipynb` file. If you then commit it, your key is public. Lesson 00.04 shows safe ways to check keys.

### Shell commands with `!`

`!command` runs a command in a shell and shows its output. On **Windows that shell is `cmd.exe`, not Git Bash**, so stick to commands that exist everywhere (like `git`) or use Python's `subprocess` (Lesson 00.02).

In [ ]:
!git --version

---
## 6. Rich output: `display()` and the last expression

Jupyter shows the value of the **last expression** in a cell automatically. `print()` shows text; `display()` shows the richest available format (tables, Markdown, images). A trailing semicolon `;` hides the last expression's output.

In [ ]:
from IPython.display import Markdown, display

display(Markdown("**Rendered Markdown** from code: handy for reports, e.g. `| model | latency |` tables."))
{"model": "openai/gpt-oss-20b", "verified": "2026-10-11"}   # Last expression → shown automatically.

In [ ]:
len("this value is computed but hidden because of the semicolon");

---
## 7. Reading errors (tracebacks)

When a cell fails, Jupyter prints a **traceback**: the chain of calls that led to the error. Read it **from the bottom up**: the last line says *what* went wrong (`KeyError: 'GROQ_API_KEY'`), the lines above say *where*.

We catch the error here so Restart & Run All keeps going; in your own experiments you will usually just let it fail.

In [ ]:
import traceback

settings = {"MASTERY_MODEL_FAST": "openai/gpt-oss-20b"}

try:
    settings["GROQ_API_KEY"]          # This key is not in the dict.
except KeyError:
    traceback.print_exc(limit=1)      # Print the traceback without stopping the notebook.
    print("\n👆 Last line = what went wrong. Fix: check the name, or use settings.get('GROQ_API_KEY').")

---
## 8. What is inside an `.ipynb` file?

A notebook is a **JSON text file**: a list of cells, each with its source and (for code cells) its **outputs**, plus metadata such as the kernel name ([nbformat docs](https://nbformat.readthedocs.io/en/latest/format_description.html)). Let's read this very notebook from disk.

*Save first* (**Ctrl + S**) so the file on disk matches what you see.

In [ ]:
import json
from collections import Counter

NOTEBOOK_FILE = Path("00-03-jupyter-fundamentals.ipynb")  # The kernel starts in the notebook's folder.

if NOTEBOOK_FILE.is_file():
    nb = json.loads(NOTEBOOK_FILE.read_text(encoding="utf-8"))
    print("nbformat version :", f"{nb['nbformat']}.{nb['nbformat_minor']}")
    print("kernel           :", nb["metadata"].get("kernelspec", {}).get("name"))
    print("cells by type    :", dict(Counter(cell["cell_type"] for cell in nb["cells"])))
    with_outputs = sum(1 for cell in nb["cells"] if cell.get("outputs"))
    print("code cells with saved outputs:", with_outputs)
else:
    print(f"{NOTEBOOK_FILE} not found in {Path.cwd()}. Did you rename or move the notebook?")

### What just happened?

- The file stores your **outputs** next to your code. Anything a cell printed (including a secret, by accident) is saved in the file.
- Outputs also make git diffs huge and noisy, and they can be stale (produced by code you have since changed).

**So the course rule is: clear outputs before committing.**

1. **Kernel → Restart Kernel and Run All Cells…** (does it work?)
2. **Edit → Clear Outputs of All Cells** (or **Kernel → Restart Kernel and Clear Outputs of All Cells…**; or find "Clear Outputs" in the command palette, **Ctrl + Shift + C**)
3. **Ctrl + S** (save)

From Git Bash, with `(lc-mastery)` active, the same in one command: `jupyter nbconvert --clear-output --inplace path/to/notebook.ipynb`. `nbconvert` comes with the course's `[dev]` install (`python -m pip install -e ".[dev]"`); if you see `Jupyter command jupyter-nbconvert not found`, install it in the active env with `python -m pip install nbconvert`.

### Checkpoints (Jupyter's safety net)

Jupyter saves a copy of the notebook in a hidden `.ipynb_checkpoints/` folder when you save. **File → Revert to Checkpoint…** restores it (the command palette finds it as "Revert to Checkpoint"; menu wording can differ slightly between versions). It is a single undo point, not version control: git is your real history. (`.ipynb_checkpoints/` is in the course `.gitignore`.)

### Bonus: strip outputs automatically with `nbstripout`

`nbstripout` installs a *git filter* so outputs are removed from what you **commit**, while your working copy keeps them ([nbstripout](https://github.com/kynan/nbstripout)). In **Git Bash**, with `(lc-mastery)` active, in the repo root:

```bash
python -m pip install nbstripout
nbstripout --install      # writes the filter to this repo's .git/config (not committed)
nbstripout --status       # should say the filter is installed
```

Undo with `nbstripout --uninstall`. It is optional now and becomes part of the tooling in Module 12. The course's unit tests also fail if a committed notebook still has outputs (`tests/unit/test_notebook_hygiene.py`).

---
## ⚠️ Pitfalls

| Symptom | Cause | Fix |
|---|---|---|
| `NameError: name 'x' is not defined` after reopening a notebook | Variables are not saved; the kernel starts empty | Run the cells above (or Restart & Run All) |
| Works for you, fails for others | Hidden state, out-of-order runs | Restart & Run All before sharing |
| Cell shows `[*]` forever | Still running or waiting for `input()` | Interrupt (**I, I**). Check for an input box below the cell |
| Package installed but `ModuleNotFoundError` | Installed in another environment | Install with `%pip` from the notebook (Lesson 00.07) and restart the kernel |
| Notebook output contains a key | Printed a secret, or ran bare `%env` | Clear outputs, **rotate the key** in the provider console, never print secrets |

## ⚠️ Production notes

> ⚠️ **Production note: notebooks are for exploring and teaching, not for running production code.**
> Hidden state and manual execution make notebooks fragile as a runtime. In production: move reusable code into a package (`src/mastery`), cover it with tests (`tests/unit/`), and run it as a service or job. Lesson 00.04 starts that habit; Modules 12–13 complete it.

## 🏋️ Exercises

1. **(Easy)** Add a Markdown cell with a heading, a bullet list, and an inline formula such as `$t_{total} = t_{in} + t_{out}$`.
2. **(Easy)** Use `%%time` on a cell that sleeps for 1.5 seconds. Compare *Wall time* with *CPU times*. Why is CPU time near zero?
3. **(Medium)** Make the out-of-order bug impossible: rewrite the `price` cell as a function `price_after(discount: float) -> float` and call it with an explicit argument.
4. **(Stretch)** Using the JSON from Section 8, count how many **lines of code** are in this notebook's code cells.

<details>
<summary><b>Show solutions</b></summary>

**2.** `time.sleep()` waits without using the CPU. Network calls to a model behave the same way: high wall time, low CPU time. That is why async code (Lesson 13.03) helps.

**3.**
```python
def price_after(discount: float, base: float = 100.0) -> float:
    """Price after applying a fractional discount (0.10 = 10 % off)."""
    return base * (1 - discount)

print(price_after(0.10))   # Always 90.0: no hidden state.
```

**4.**
```python
sum(len(cell["source"].splitlines()) for cell in nb["cells"] if cell["cell_type"] == "code")
```
</details>

## ✅ Checkpoint

- [ ] You can add, delete, undo, and change cells with the keyboard.
- [ ] You can explain why `price` printed 50.00 and how Restart & Run All fixes it.
- [ ] You know which magic leaks secrets (bare `%env`) and how to clear outputs.
- [ ] **Do it now:** Restart & Run All this notebook, then clear outputs and save.

## 🔎 Research checkpoint

Open [JupyterLab commands and shortcuts](https://jupyterlab.readthedocs.io/en/stable/user/commands.html). Find one command that has **no** keyboard shortcut but that you would use often (hint: look at "Restart Kernel and Run All Cells…"). Note it in `docs/research-log.md`. You can add your own shortcut in *Settings → Settings Editor → Keyboard Shortcuts*.

## Summary

- The kernel keeps state; the page does not. Execution order is what counts.
- **Restart Kernel and Run All Cells** is the test; clear outputs before every commit.
- `%time`, `%%time`, `%timeit`, `%whos`, `?` are daily tools. Bare `%env` is dangerous.
- On Windows, `!` runs in `cmd.exe`.

**Next:** [00.04 · Secrets and `.env` hygiene](00-04-secrets-and-env.ipynb).

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 00.04 · Secrets and `.env` hygiene** → [`00-04-secrets-and-env.ipynb`](00-04-secrets-and-env.ipynb).

[Module 00 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)